In [5]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer,AutoModelForCausalLM

# Use tokenizer cl100k_base (used by GPT-4) 

In [10]:
enc = tiktoken.get_encoding("cl100k_base")
print(f"{enc.n_vocab} tokens in the encoding")

100277 tokens in the encoding


# Load and encode sample text

In [12]:
with open("sample_text.txt", "r") as f:
    text = f.read()

enc_text = enc.encode(text)
print(f"Encoded text length: {len(enc_text)}")

Encoded text length: 304


# Use allowed_special to include special tokens

In [18]:
enc.encode("Hello<|endoftext|>world", allowed_special={"<|endoftext|>"})


[9906, 100257, 14957]

# Create torch dataset for next token prediction
- max_length: length of the input sequence
- stride: skip size for overlapping sequences

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, text,tokenizer,max_length,stride):
        self.tokenizer = tokenizer
        self.text = text
        self.max_length = max_length
        self.stride = stride
        
        self.input_ids = []
        self.target_ids = []
        
        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            input_ids = token_ids[i:i + max_length]
            target_ids = token_ids[i + 1:i + max_length + 1]
            
            self.input_ids.append(input_ids)
            self.target_ids.append(target_ids)
        
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        x = torch.tensor(self.input_ids[idx], dtype=torch.long)
        y = torch.tensor(self.target_ids[idx], dtype=torch.long)
        return x, y

# Load the dataset

In [76]:
dataloader = DataLoader(CustomDataset(text, enc, max_length=4, stride=4), batch_size=1, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
input_ids, target_ids = first_batch

In [77]:
print(f"shape input_ids: {input_ids.shape} -> batch_size x sequence_length")
input_ids

shape input_ids: torch.Size([1, 4]) -> batch_size x sequence_length


tensor([[  791, 63479,  1646,   374]])

# Create Embedding Layer which is learned during training

In [91]:
vicab_size = enc.n_vocab
print(f"Vocab size: {vicab_size}") 

embed_dims = 8 
print(f"Output dims: {embed_dims}")

token_embed = torch.nn.Embedding(vicab_size, embed_dims)
print(f"Token embedding shape: {token_embed.weight.shape}")

Vocab size: 100277
Output dims: 8
Token embedding shape: torch.Size([100277, 8])


In [92]:
token_embed(input_ids).shape

torch.Size([1, 4, 8])

# Decoded inputs and outputs

In [93]:
enc.decode(input_ids[0].tolist()), enc.decode(target_ids[0].tolist())
print(f"Input IDs: {input_ids[0]}")
print(f"decoded input: {enc.decode(input_ids[0].tolist())}")

print(f"Target IDs: {target_ids[0]}")
print(f"decoded target: {enc.decode(target_ids[0].tolist())}")

Input IDs: tensor([  791, 63479,  1646,   374])
decoded input: The Transformer model is
Target IDs: tensor([63479,  1646,   374,   264])
decoded target:  Transformer model is a


# Asuming no positional embedding 
input to transformer block

In [94]:
inputs = token_embed(input_ids)
print(f"Input shape: {inputs.shape} -> batch_size x sequence_length x embedding_dim")
inputs

Input shape: torch.Size([1, 4, 8]) -> batch_size x sequence_length x embedding_dim


tensor([[[-0.6567, -0.3956,  0.3216, -2.0957, -1.0561, -0.3409, -1.0131,
          -1.8803],
         [ 0.7112, -0.0663, -0.1428, -0.3361, -0.1008,  0.6043,  0.1178,
          -0.4921],
         [-0.1211,  0.3232, -1.4237, -0.0386,  0.1451,  1.4693, -0.4910,
           2.2244],
         [-0.1578,  0.7147,  1.4045, -0.6522, -0.7258,  0.6495, -0.3265,
          -0.5020]]], grad_fn=<EmbeddingBackward0>)

# Self attention

In [96]:
d_in = inputs.shape[1]
d_out = 8 # dimensions of the output 

print(f"Input shape: {inputs.shape} -> batch_size x sequence_length x embedding_dim")
print(f"Output shape: {inputs.shape} -> batch_size x sequence_length x embedding_dim")

Query_w = torch.nn.Linear(d_in, d_out, bias=False)
Key_w = torch.nn.Linear(d_in, d_out, bias=False)
Value_w = torch.nn.Linear(d_in, d_out, bias=False)
print(f"Query weight shape: {Query_w.weight.shape} -> input_dim x output_dim")
print(f"Key weight shape: {Key_w.weight.shape} -> input_dim x output_dim")
print(f"Value weight shape: {Value_w.weight.shape} -> input_dim x output_dim")

Input shape: torch.Size([1, 4, 8]) -> batch_size x sequence_length x embedding_dim
Output shape: torch.Size([1, 4, 8]) -> batch_size x sequence_length x embedding_dim
Query weight shape: torch.Size([8, 4]) -> input_dim x output_dim
Key weight shape: torch.Size([8, 4]) -> input_dim x output_dim
Value weight shape: torch.Size([8, 4]) -> input_dim x output_dim


# Attention weights for self attention

In [104]:
query = inputs@Query_w.weight
key = inputs@Key_w.weight
value = inputs@Value_w.weight

attention_scores = query @ key.transpose(1, 2) / (d_out ** 0.5)
attention_weights = torch.nn.functional.softmax(attention_scores, dim=-1)
attention_weights

tensor([[[0.1388, 0.1701, 0.5466, 0.1445],
         [0.2739, 0.2516, 0.2223, 0.2522],
         [0.2575, 0.2557, 0.1773, 0.3095],
         [0.2900, 0.2136, 0.2772, 0.2192]]], grad_fn=<SoftmaxBackward0>)

# Self Attention Class

In [105]:
import torch.nn as nn

class SelfAttention(nn.Module):
    def __init__(self, d_in,d_out):
        super(SelfAttention, self).__init__()
        self.Query_w = nn.Linear(d_in, d_out, bias=False)
        self.Key_w = nn.Linear(d_in, d_out, bias=False)
        self.Value_w = nn.Linear(d_in, d_out, bias=False)
        self.d_out = d_out
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x):
        query = x @ self.Query_w.weight
        key = x @ self.Key_w.weight
        value = x @ self.Value_w.weight
        
        attention_scores = query @ key.transpose(1, 2) / (self.d_out ** 0.5)
        attention_weights = self.softmax(attention_scores)
        context_vector = attention_weights @ value
        return context_vector

# Get Context Vectors

In [ ]:
self_attention = SelfAttention(d_in, d_out)
context_vector = self_attention(inputs)
context_vector

tensor([[[ 0.0273,  0.2255, -0.5316, -0.1683],
         [-0.0043,  0.2053, -0.2778,  0.0737],
         [ 0.0119,  0.1944, -0.1518,  0.1994],
         [-0.0084,  0.2147, -0.2931,  0.0793]]], grad_fn=<UnsafeViewBackward0>)